# Experiment: Case 001 Molecular Test Escalation Gate

Objective: convert the `phenotype_only` label into a ranked molecular-record request sequence.

Success criteria:
- no treatment advice;
- no patient identifiers or raw records;
- each request maps to a source-backed reason;
- the output tells the repo when case-001 can move from `phenotype_only` to stronger labels.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

CURRENT_LABEL = "phenotype_only"
CURRENT_LABEL

## Ranking Model

The score ranks record requests by how much they reduce uncertainty. It is not a diagnosis score and does not tell anyone to order a test without clinician review.


In [ ]:
@dataclass(frozen=True)
class MolecularRecord:
    """A molecular or hemoglobin-context record needed for case routing."""

    name: str
    request: str
    closes_label: str
    subtype_value: int
    hbf_value: int
    globin_balance_value: int
    availability_first: int
    source_anchor: str

    def score(self) -> int:
        """Rank by case-routing value and practical first-step usefulness."""
        return (
            self.subtype_value
            + self.hbf_value
            + self.globin_balance_value
            + self.availability_first
        )


records = [
    MolecularRecord(
        name="historical_hplc_transfusion_proximity",
        request=(
            "Ask whether the infancy HPLC/electrophoresis was before any transfusion."
        ),
        closes_label="phenotype_context_clearer",
        subtype_value=4,
        hbf_value=4,
        globin_balance_value=2,
        availability_first=5,
        source_anchor="GeneReviews beta-thalassemia hemoglobin analysis",
    ),
    MolecularRecord(
        name="hbb_sequence_result",
        request=(
            "Ask for HBB sequence result with variant classification and "
            "beta-zero/beta-plus interpretation."
        ),
        closes_label="hbb_confirmed",
        subtype_value=5,
        hbf_value=4,
        globin_balance_value=3,
        availability_first=4,
        source_anchor="GeneReviews beta-thalassemia molecular testing",
    ),
    MolecularRecord(
        name="hbb_deletion_duplication_result",
        request=(
            "If HBB sequencing is incomplete, ask whether HBB deletion "
            "or duplication analysis is needed."
        ),
        closes_label="hbb_confirmed_when_sequence_incomplete",
        subtype_value=4,
        hbf_value=3,
        globin_balance_value=2,
        availability_first=3,
        source_anchor="GeneReviews beta-thalassemia deletion/duplication follow-up",
    ),
    MolecularRecord(
        name="hba1_hba2_copy_number",
        request=(
            "Ask for HBA1/HBA2 common deletion and copy-number result, "
            "including alpha duplication if tested."
        ),
        closes_label="hbb_hba_confirmed",
        subtype_value=4,
        hbf_value=2,
        globin_balance_value=5,
        availability_first=4,
        source_anchor="GeneReviews alpha-thalassemia molecular testing",
    ),
    MolecularRecord(
        name="hpfh_delta_beta_or_hbg_context",
        request=(
            "Ask whether HPFH, delta-beta-thalassemia, HBG1/HBG2 "
            "promoter, or XmnI-HBG2 context was reviewed."
        ),
        closes_label="modifier_context_known",
        subtype_value=3,
        hbf_value=5,
        globin_balance_value=3,
        availability_first=2,
        source_anchor="case001 high-HbF gate and TIF diagnosis chapter",
    ),
    MolecularRecord(
        name="family_fallback_records",
        request=(
            "If patient testing is delayed or confounded, ask whether "
            "parent HPLC or genetic records can clarify inheritance."
        ),
        closes_label="family_context_partial",
        subtype_value=3,
        hbf_value=2,
        globin_balance_value=3,
        availability_first=3,
        source_anchor="GeneReviews genetic counseling and family testing",
    ),
]

ranked_records = sorted(records, key=lambda record: (-record.score(), record.name))
[(record.name, record.closes_label, record.score()) for record in ranked_records]

## Decision Output

The output is phrased as records to request from clinicians or existing hospital files. It is not an instruction to self-order testing.


In [ ]:
decision_packet = {
    "current_label": CURRENT_LABEL,
    "first_requests": [record.request for record in ranked_records[:3]],
    "second_requests": [record.request for record in ranked_records[3:]],
    "upgrade_rule": (
        "Do not upgrade beyond phenotype_only until HBB is documented. "
        "Do not treat HbF/HPFH-like interpretation as known until modifier "
        "or HPFH/delta-beta context is reviewed."
    ),
    "boundary": "Clinician or genetics review required before patient-specific claims.",
}

assert decision_packet["current_label"] == "phenotype_only"
assert len(decision_packet["first_requests"]) == 3
decision_packet

## Interpretation

The first three asks are:

1. whether the historical Hb fractions were measured before transfusion;
2. the `HBB` sequence result and beta-zero/beta-plus interpretation;
3. `HBA1/HBA2` deletion/copy-number context.

The HPFH/delta-beta/HBG question remains essential for the research lane, but it should not replace the basic `HBB` and `HBA` record requests.
